### Imports

In [24]:
import json
import os
import pandas as pd

from utils_MS import *

# %load_ext autotime

In [25]:
# def run(params):

### Parameters

In [26]:
""" try:
    dir = os.path.dirname(os.path.abspath(__file__))
except:
    dir = os.getcwd()
print(dir) """

' try:\n    dir = os.path.dirname(os.path.abspath(__file__))\nexcept:\n    dir = os.getcwd()\nprint(dir) '

In [27]:
dict_dataset = {
    1: ["Mentos_2_process_NormalizationFiltered_format", ["Orange"]], # new mentos,  for Metabolomics
    2: ["deybis_filter_september_2br_3ar_format", ["SecoAmazonas"]],
    3: ["deybis_filter_december_2br_3ar_format", ["SecoAmazonas"]],
    4: ["deybis_filter_september_2br_10ar_format", ["SecoAmazonas"]],
    5: ["deybis_filter_december_2br_10ar_format", ["SecoAmazonas"]], # for Metabolomics
    6: ["deybis_filter_september_min_2br_3ar_format", ["SecoAmazonas"]],
    7: ["deybis_filter_december_min_2br_3ar_format", ["SecoAmazonas"]],
    8: ["vanessa_december_2br_3ar_format", ["SecoAmazonas"]],
    9: ["Pablo_2br_nar_format", ["AR"]],
    10: ["Pablo_2br_14ar_format", ["AR"]],
}
dataset = dict_dataset[9] # change
dataset

['Pablo_2br_nar_format', ['AR']]

In [28]:
params = {
    "exp": "exp9", # Change
    "methods": ["t-gae"], # ["vgae-base", "argva-base", "vgae-line", "dgi-tran", "t-gae"],
    "data_variations": ["none"],
    "has_transformation": True, # True or False
    "controls": dataset[1],
    "dimension": 32,
    "threshold_corr": 0.5,
    "threshold_log2": 0,
    "alpha": 0.05,
    "iterations": 1,
    "raw_data_file": dataset[0],
    "groups_id_no": ["Blank", "QC", "Std"],
    "sensitivity": False, # False: f1 (selectivity), True: f1 (selectivity), f2 (sensitivity)
    "obs": "",
    "seeds": [41, 42, 43, 44, 45, 46],
    
    "from": "python",
    "cuda": 0,
    "epochs": 100,
    "lr": 0.0001,
    "weight_decay": 1e-4,
    "patience": 10,
    "contamination": 0.1, # float in (0., 0.5)
    "n_jobs": 1, # -1 all
}

In [29]:
""" dir_path = "experiments/output"
res = sorted(os.listdir(dir_path))
n = len(res)
exp = "exp{}".format(n) """

exp = str(params["exp"])
exp

'exp9'

### Load dataset

In [30]:
# load dataset groups
if params["from"] == "python":
    df_raw = pd.read_csv("experiments/raw_data/{}.csv".format(params["raw_data_file"]), delimiter="|")
elif params["from"] == "drf":
    df_raw = pd.read_csv("{}".format(params["raw_data_file"]), delimiter="|") # from DRF
df_raw

,Alignment ID,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,0,1.0,69.99951,Unknown,0.575329,-2.215760,-1.877019,-1.518844,-1.398838,-1.298920,...,-1.096866,0.333022,0.273693,0.478082,-1.012425,-1.078037,0.566607,-1.074093,-1.041128,-1.076718
1,1,1.0,70.04025,Unknown,2.519698,0.524517,0.862641,1.220163,1.339951,1.439686,...,1.992662,2.054098,2.342706,2.060094,2.337721,2.048076,2.197014,2.063083,2.188817,2.051090
2,2,1.0,70.04151,Unknown,0.580982,-0.489549,-0.214833,0.075646,0.172970,0.254003,...,0.520551,0.533229,0.582630,0.534187,0.581785,0.532269,0.560055,0.535142,0.559160,1.820908
3,3,1.0,70.04908,Unknown,2.570505,-0.107955,0.215094,0.556676,0.671123,0.766412,...,1.365489,0.697392,2.546915,0.700091,2.183391,0.694677,2.494512,0.701435,0.730124,0.696036
4,4,1.0,70.06267,Unknown,1.157874,-0.151374,0.127539,0.297939,0.422454,0.521265,...,1.125370,1.149314,2.839522,2.768711,1.226690,1.147188,1.188020,2.990083,1.187039,1.148252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,5439,1.0,732.79951,Unknown,2.214770,-0.588492,-0.389296,-0.178671,-0.108101,4.236125,...,0.103094,0.175553,0.453542,0.181423,0.451090,0.169659,0.318208,0.184348,0.310187,0.172609
5440,5440,1.0,748.76437,Unknown,0.725796,-1.431324,-1.052855,-0.652672,-0.518591,-0.406954,...,0.197692,0.275529,0.580224,0.281838,0.574945,0.269194,0.431907,0.284983,0.423283,0.272365
5441,5441,1.0,794.79590,Unknown,1.651239,-1.204281,-0.800512,-0.373577,-0.230533,-0.111434,...,0.318278,0.382511,0.619537,0.388773,0.614171,0.376221,2.624637,0.391893,0.511456,0.379369
5442,5442,1.0,800.81295,Unknown,1.511482,-1.036480,-0.607684,-0.154288,-0.002378,0.124103,...,0.754284,0.812592,1.028620,1.608131,1.024227,0.807407,0.923866,0.817754,0.919121,0.810002


### Format dataset

In [31]:
# has transformation
columns_data = list(df_raw.columns)[4:]
if params["has_transformation"]:
    print("transformation")
    for column in columns_data:
        df_raw[column] = df_raw[column].apply(lambda x: 10**x)
df_raw

transformation


,Alignment ID,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,0,1.0,69.99951,Unknown,3.761221,0.006085,0.013273,3.028000e-02,0.039917,0.050244,...,0.080008,2.152889,1.877989,3.006645,0.097180,0.083553,3.686441,0.084316,0.090964,0.083807
1,1,1.0,70.04025,Unknown,330.900909,3.345932,7.288550,1.660211e+01,21.875132,27.522388,...,98.324448,113.265571,220.143630,114.840334,217.630894,111.705774,157.403484,115.633361,154.460280,112.483808
2,2,1.0,70.04151,Unknown,3.810500,0.323930,0.609772,1.190270e+00,1.489259,1.794746,...,3.315511,3.413731,3.824988,3.421265,3.817551,3.406194,3.631237,3.428796,3.623766,66.207625
3,3,1.0,70.04908,Unknown,371.967656,0.779912,1.640943,3.603099e+00,4.689464,5.839993,...,23.200059,4.981860,352.301595,5.012925,152.542561,4.950816,312.257226,5.028466,5.371851,4.966335
4,4,1.0,70.06267,Unknown,14.383827,0.705709,1.341339,1.985815e+00,2.645173,3.320973,...,13.346583,14.103080,691.069505,587.099070,16.853497,14.034201,15.417710,977.423380,15.382943,14.068638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,5439,1.0,732.79951,Unknown,163.972145,0.257934,0.408042,6.627189e-01,0.779649,17223.650862,...,1.267927,1.498143,2.841460,1.518527,2.825464,1.477947,2.080693,1.528791,2.042619,1.488022
5440,5440,1.0,748.76437,Unknown,5.318578,0.037040,0.088541,2.224989e-01,0.302977,0.391783,...,1.576491,1.885946,3.803859,1.913544,3.757898,1.858634,2.703381,1.927451,2.650225,1.872254
5441,5441,1.0,794.79590,Unknown,44.795976,0.062477,0.158303,4.230803e-01,0.588121,0.773688,...,2.081027,2.412741,4.164254,2.447782,4.113120,2.378048,421.344464,2.465433,3.246804,2.395351
5442,5442,1.0,800.81295,Unknown,32.469955,0.091943,0.246783,7.009899e-01,0.994539,1.330769,...,5.679157,6.495188,10.681201,40.563104,10.573695,6.418109,8.392011,6.572858,8.300825,6.456575


In [32]:
# concat
df_join_raw = pd.concat([
    df_raw.iloc[:, :]], axis=1)
df_join_raw.set_index("Alignment ID", inplace=True)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
Alignment ID,,,,,,,,,,,,,,,,,,,,,
0,1.0,69.99951,Unknown,3.761221,0.006085,0.013273,3.028000e-02,0.039917,0.050244,0.061226,...,0.080008,2.152889,1.877989,3.006645,0.097180,0.083553,3.686441,0.084316,0.090964,0.083807
1,1.0,70.04025,Unknown,330.900909,3.345932,7.288550,1.660211e+01,21.875132,27.522388,33.526282,...,98.324448,113.265571,220.143630,114.840334,217.630894,111.705774,157.403484,115.633361,154.460280,112.483808
2,1.0,70.04151,Unknown,3.810500,0.323930,0.609772,1.190270e+00,1.489259,1.794746,2.106842,...,3.315511,3.413731,3.824988,3.421265,3.817551,3.406194,3.631237,3.428796,3.623766,66.207625
3,1.0,70.04908,Unknown,371.967656,0.779912,1.640943,3.603099e+00,4.689464,5.839993,7.051657,...,23.200059,4.981860,352.301595,5.012925,152.542561,4.950816,312.257226,5.028466,5.371851,4.966335
4,1.0,70.06267,Unknown,14.383827,0.705709,1.341339,1.985815e+00,2.645173,3.320973,4.013620,...,13.346583,14.103080,691.069505,587.099070,16.853497,14.034201,15.417710,977.423380,15.382943,14.068638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1.0,732.79951,Unknown,163.972145,0.257934,0.408042,6.627189e-01,0.779649,17223.650862,0.892598,...,1.267927,1.498143,2.841460,1.518527,2.825464,1.477947,2.080693,1.528791,2.042619,1.488022
5440,1.0,748.76437,Unknown,5.318578,0.037040,0.088541,2.224989e-01,0.302977,0.391783,0.488620,...,1.576491,1.885946,3.803859,1.913544,3.757898,1.858634,2.703381,1.927451,2.650225,1.872254
5441,1.0,794.79590,Unknown,44.795976,0.062477,0.158303,4.230803e-01,0.588121,0.773688,0.979274,...,2.081027,2.412741,4.164254,2.447782,4.113120,2.378048,421.344464,2.465433,3.246804,2.395351


In [33]:
# split
df_join_raw = df_join_raw.rename_axis(None)
# df_join_raw = df_join_raw.iloc[:, 2:]
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.43,PD_2.44,PD_2.45,PD_2.46,PD_2.47,PD_2.48,PD_2.49,PD_2.50,PD_2.51,PD_2.52
0,1.0,69.99951,Unknown,3.761221,0.006085,0.013273,3.028000e-02,0.039917,0.050244,0.061226,...,0.080008,2.152889,1.877989,3.006645,0.097180,0.083553,3.686441,0.084316,0.090964,0.083807
1,1.0,70.04025,Unknown,330.900909,3.345932,7.288550,1.660211e+01,21.875132,27.522388,33.526282,...,98.324448,113.265571,220.143630,114.840334,217.630894,111.705774,157.403484,115.633361,154.460280,112.483808
2,1.0,70.04151,Unknown,3.810500,0.323930,0.609772,1.190270e+00,1.489259,1.794746,2.106842,...,3.315511,3.413731,3.824988,3.421265,3.817551,3.406194,3.631237,3.428796,3.623766,66.207625
3,1.0,70.04908,Unknown,371.967656,0.779912,1.640943,3.603099e+00,4.689464,5.839993,7.051657,...,23.200059,4.981860,352.301595,5.012925,152.542561,4.950816,312.257226,5.028466,5.371851,4.966335
4,1.0,70.06267,Unknown,14.383827,0.705709,1.341339,1.985815e+00,2.645173,3.320973,4.013620,...,13.346583,14.103080,691.069505,587.099070,16.853497,14.034201,15.417710,977.423380,15.382943,14.068638
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1.0,732.79951,Unknown,163.972145,0.257934,0.408042,6.627189e-01,0.779649,17223.650862,0.892598,...,1.267927,1.498143,2.841460,1.518527,2.825464,1.477947,2.080693,1.528791,2.042619,1.488022
5440,1.0,748.76437,Unknown,5.318578,0.037040,0.088541,2.224989e-01,0.302977,0.391783,0.488620,...,1.576491,1.885946,3.803859,1.913544,3.757898,1.858634,2.703381,1.927451,2.650225,1.872254
5441,1.0,794.79590,Unknown,44.795976,0.062477,0.158303,4.230803e-01,0.588121,0.773688,0.979274,...,2.081027,2.412741,4.164254,2.447782,4.113120,2.378048,421.344464,2.465433,3.246804,2.395351
5442,1.0,800.81295,Unknown,32.469955,0.091943,0.246783,7.009899e-01,0.994539,1.330769,1.709164,...,5.679157,6.495188,10.681201,40.563104,10.573695,6.418109,8.392011,6.572858,8.300825,6.456575


In [34]:
# get groups name
groups_id_no = params["groups_id_no"]
groups_id = []
for item in df_join_raw.iloc[:, 3:].columns.values:
    group_id = item.split("_")[0]
    if group_id not in groups_id and group_id not in groups_id_no:
        groups_id.append(group_id)
groups_id

['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD']

In [35]:
# delete no sample columns
""" columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]
df_join_raw.drop(columns_delete, axis=1, inplace=True)
df_join_raw """

' columns_delete = [columna for columna in df_join_raw.columns if columna.split("_")[0] in columns_no_sample]\ndf_join_raw.drop(columns_delete, axis=1, inplace=True)\ndf_join_raw '

In [36]:
# get subgroups names

""" def get_subgroups_id(df_join_raw, groups, by_group=False):
    dict_groups_id = {}
    for group in groups:
        # get group
        if by_group:
            dict_groups_id[group] = ["1"]
        else:
            columns = list(df_join_raw.filter(like=group).columns)
            subgroups = [item.split("{}_".format(group))[1].split(".")[0] for item in columns]
            subgroups = np.unique(subgroups)
            dict_groups_id[group] = subgroups.tolist()
    return dict_groups_id """

subgroups_id = get_subgroups_id(df_join_raw, groups_id)
subgroups_id

{'AR': ['1', '2'],
 'CRS': ['1', '2'],
 'OSA': ['1', '2'],
 'LPRD': ['1', '2'],
 'SGB': ['1', '2'],
 'LSNB': ['1', '2'],
 'RCC': ['1', '2'],
 'BC': ['1', '2'],
 'BPH': ['1', '2'],
 'PCa': ['1', '2'],
 'PD': ['1', '2']}

In [37]:
# count analtical repetitions
# df_join_raw.filter(like="AA_1.")

In [38]:
# check distribution

In [39]:
""" x = df_join_raw.iloc[2, 3:]
print(x.min(), x.max(), x.mean())
x.hist(bins=200) """

' x = df_join_raw.iloc[2, 3:]\nprint(x.min(), x.max(), x.mean())\nx.hist(bins=200) '

In [40]:
# f_join_raw.iloc[:, 5].hist(bins=100)

In [41]:
params["controls"], groups_id

(['AR'],
 ['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD'])

In [42]:
# get groups combination
groups = []
controls = params["controls"]

groups = []
for control in controls:
    for group_id in groups_id:
        if control != group_id:
            groups.append([control, group_id])
print(groups)

[['AR', 'CRS'], ['AR', 'OSA'], ['AR', 'LPRD'], ['AR', 'SGB'], ['AR', 'LSNB'], ['AR', 'RCC'], ['AR', 'BC'], ['AR', 'BPH'], ['AR', 'PCa'], ['AR', 'PD']]


### Create folders

In [43]:
# create experiments folder
try: 
    os.mkdir("experiments/output/{}".format(exp))
    os.mkdir("experiments/output/{}/correlations".format(exp))
    os.mkdir("experiments/output/{}/preprocessing".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/edges".format(exp))
    os.mkdir("experiments/output/{}/preprocessing/graphs_data".format(exp))
    os.mkdir("experiments/output/{}/loss".format(exp))
    os.mkdir("experiments/output/{}/node_embeddings".format(exp))
    os.mkdir("experiments/output/{}/common_nodes".format(exp))
    os.mkdir("experiments/output/{}/filter_raw".format(exp))
    os.mkdir("experiments/output/{}/plots".format(exp))
except OSError as error: 
    print(error)

### Save dataset and parameters

In [44]:
# save dataset
df_join_raw.to_csv("experiments/input/{}_raw.csv".format(exp), index=True)

# save parameters
parameters = {
    "exp": exp,
    "methods": params["methods"],
    "data_variations": params["data_variations"],
    "has_transformation": params["has_transformation"],
    "controls": params["controls"],
    "dimension": params["dimension"],
    "threshold_corr": params["threshold_corr"],
    "threshold_log2": params["threshold_log2"],
    "alpha": params["alpha"],
    "iterations": params["iterations"],
    "raw_data_file": params["raw_data_file"],
    "groups_id": groups_id,
    "subgroups_id": subgroups_id,
    "groups": groups,
    "groups_id_no": params["groups_id_no"],
    "sensitivity": params["sensitivity"],
    
    "from": params["from"],
    "cuda": params["cuda"],
    "epochs": params["epochs"],
    "lr": params["lr"],
    "weight_decay": params["weight_decay"],
    "patience": params["patience"],
    "contamination": params["contamination"],
    "n_jobs": params["n_jobs"],

    "seeds": params["seeds"],
    "obs": params["obs"]
}

with open("experiments/output/{}/parameters.json".format(exp), "w") as outfile:
    json.dump(parameters, outfile, indent=4)

In [45]:
experiments = {
    "exp": exp
}

with open("exp.json".format(experiments), "w") as outfile:
    json.dump(experiments, outfile, indent=4)

In [46]:
# return exp